# Lab 4: Relationship Extraction
*   **Course**: CM52065 Natural Language Processing
*   **Author**: Dr. Andrew Barnes

## Overview
This week we will be using spaCy to perform POS and NER tagging. We will then follow this up by looking at how to implement a basic relationship extraction routine.

The learning objectives for this lab are as follows:

1. Apply POS and NER tagging on real-world use-cases.
2. Visualise POS and NER tags.
3. Explain the limitations of POS, NER and RE methods.

To achieve these objectives we will be using a corpus of BBC News Articles.

## Dataset

The dataset provided for this lab contains BBC news articles.

### 1.1 Loading the dataset
We do not need to upload any datasets for this lab, instead we willl make use of the `datasets` package and load articles from there.

> NOTE: This dataset has been limited to 100 articles for speed purposes, feel free to choose a different 100 or increase the number of articles.

In [1]:
from datasets import load_dataset
ds = load_dataset("permutans/c4-bbc-news")['train']['text'][:100]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
ds[0]

'The Rolling Stones\' US tour is likely to take place in July, following news that Mick Jagger had to postpone 17 dates due to ill health.\nThe band are working with promoters to reschedule the shows, amid reports that Jagger will have heart surgery later this week.\n"I really hate letting you down like this," tweeted the star after the tour was postponed at the weekend.\n"I will be working very hard to be back on stage as soon as I can."\nUS gossip website Drudge Report was the first to report that Jagger would need surgery to replace a heart valve. The story was subsequently confirmed by US music magazine Rolling Stone.\nThe 75-year-old is expected to make a full recovery and return to touring this summer.\n"We\'re beginning to look at the rescheduling options and we\'re going to try and do this as quickly as we can," said John Meglen, of the Stones\' promoters Concerts West.\n"Everyone\'s health and happiness comes first," he told Billboard, adding that new dates could be announced 

Let's take a look at a few articles:

In [3]:
print(f"Article 20: {ds[20]}\n\n")
print(f"Article 200: {ds[60]}\n\n")
print(f"Article 20000: {ds[90]}")

Article 20: A year on from the Brexit referendum, says the Financial Times, the government has still not spelled out what that will mean for the economy.
The paper sees division in the two main parties, the House of Lords, and across the UK.
If things turn nasty, it thinks the government should resist the "petulant and reckless" option of walking out.
But the Sun tells Theresa May "the time for niceties is over."
It says the PM has now assured every EU ­citizen here that they can stay, come what may - and it's time for other EU leaders to be "equally forthcoming".
And four former Conservative cabinet ministers tell the Daily Telegraph that she should walk away if the EU won't move on to discussions about trade and the future.
The Daily Mail and the Daily Express both see signs that Germany, at least, might want a comprehensive free trade accord.
Several of the papers are struck by - and concerned about - the figures showing how many people are financially exposed.
Millions of people, s

## Part 1: Preprocessing

As you are hopefully getting used to by now we are going to begin by pre-processing the artciles by performing cleaning and tokenisation. We do not need to do any embedding in this lab.

I have provided some pre-processing code below for you.

> **Question:** Why doesn't the code below remove stop words or punctuation?

You will also notice we do not extract just the text components from the tokens as we did in previous labs. This is because we are interested in more than just the text this time!

In [4]:
from tqdm import tqdm
import spacy

# Load small model
nlp = spacy.load("en_core_web_sm")

def tokenise(article):
  processed_doc = nlp(article)                                     # Conversion
  return processed_doc

Now let's perform preprocessing on all articles:

In [5]:
def preprocess(articles):
  processed_articles = []
  for article in tqdm(articles):
    processed_articles.append(tokenise(article))
  return processed_articles
processed_articles = preprocess(ds)

100%|██████████| 100/100 [00:09<00:00, 10.64it/s]


Let's take a look at a before and after for an article:

In [6]:
print(f"Before: {ds[80]}\n\n")
print(f"After: {processed_articles[80]}")

Before: Japan has moved closer to lifting a two-year ban on US and Canadian beef after government health advisors said the meat posed a 'slim' risk to consumers.
A sub-committee of Japan's Food Safety Commission said imports, which had been halted amid fears of mad-cow disease, could resume as early as December.
The ban has proved controversial and the US threatened retaliatory tariffs.
to Japan on 15 and 16 November.
The ban on US beef was imposed in December 2003, seven months after halting imports from Canada.
At the end of 2004, it seemed as if a breakthrough had been negotiated when Japan agreed to exempt cattle that was younger than 20 months.
Japan attached conditions to the deal and it pretty much stalled, prompting increased frustration among US producers.
Before the ban, Japan was the biggest importer of US beef.
Speaking on Monday, the chairman of the committee that looked into the risks related to US beef said that a ban could be lifted as long as dangerous body parts - suc

## Part 2: Part-of-speech Tagging

Now we have our processed corpus we can extract and visualise the POS tags allocated through the spaCy module. spaCy uses the underlying model (in this case we used `en_core_web_sm`) to tag POS and a range of additional linguistic features as illustrated in the documentation [here](https://spacy.io/usage/linguistic-features).

### Tagging

Before we jump in with applying these breakdowns to our articles lets start with a simple sentence to illustrate how we can apply POS tags.

In [7]:
text = "Julie started working at Google in 2015."
tokenised = tokenise(text)

First, let's look at how this sentence is tokenised, this should be familiar to you:

Next, let's take a look at the POS tags which have been allocated to each unit.

In [8]:
for token in tokenised:
  print(f"{token.text:12} | POS: {token.pos_:6} | Detailed POS: {token.tag_}")
  # https://github.com/explosion/spaCy/blob/master/spacy/glossary.py
  # "NNP": "noun, proper singular"
  # "VBD": "verb, past tense"
  # "VBG": "verb, gerund or present participle"
  # "CD": "cardinal number",

Julie        | POS: PROPN  | Detailed POS: NNP
started      | POS: VERB   | Detailed POS: VBD
working      | POS: VERB   | Detailed POS: VBG
at           | POS: ADP    | Detailed POS: IN
Google       | POS: PROPN  | Detailed POS: NNP
in           | POS: ADP    | Detailed POS: IN
2015         | POS: NUM    | Detailed POS: CD
.            | POS: PUNCT  | Detailed POS: .


Notice there are two levels of tags provided here, the initial POS tags and the Detailed POS tags. The detailed tags are useful for relationship extraction.

If you need a reminder as to what a POS/Tag means you can call `spacy.explain('TAG')` such as the example below:

In [9]:
spacy.explain('NNP')

'noun, proper singular'

Using the space below output the POS tags for one of the articles provided in the dataset.

> TIP: If you want to make this more manageable consider taking one sentence of an article. Large documents can be broken into sentences in the same way we can tokenise them, to do this use spaCy's `Sentencizer`, guidance [here](https://spacy.io/api/sentencizer).



> **Question:** do you notice any trends in the distribution of POS tags?

In [10]:
### BEGIN CODE

nlp.add_pipe("sentencizer")

doc = nlp(processed_articles[80])
tokenised_text = tokenise(list(doc.sents)[0].text)

for token in tokenised_text:
  print(f"{token.text:12} | POS: {token.pos_:6} | Detailed POS: {token.tag_}")
### END CODE

Japan        | POS: PROPN  | Detailed POS: NNP
has          | POS: AUX    | Detailed POS: VBZ
moved        | POS: VERB   | Detailed POS: VBN
closer       | POS: ADV    | Detailed POS: RBR
to           | POS: ADP    | Detailed POS: IN
lifting      | POS: VERB   | Detailed POS: VBG
a            | POS: DET    | Detailed POS: DT
two          | POS: NUM    | Detailed POS: CD
-            | POS: PUNCT  | Detailed POS: HYPH
year         | POS: NOUN   | Detailed POS: NN
ban          | POS: NOUN   | Detailed POS: NN
on           | POS: ADP    | Detailed POS: IN
US           | POS: PROPN  | Detailed POS: NNP
and          | POS: CCONJ  | Detailed POS: CC
Canadian     | POS: ADJ    | Detailed POS: JJ
beef         | POS: NOUN   | Detailed POS: NN
after        | POS: SCONJ  | Detailed POS: IN
government   | POS: NOUN   | Detailed POS: NN
health       | POS: NOUN   | Detailed POS: NN
advisors     | POS: NOUN   | Detailed POS: NNS
said         | POS: VERB   | Detailed POS: VBD
the          | POS: DET 

### Dependencies

One final thing spaCy allows us to do is visualise the dependencies between the words in a sentence as shown below.

In [11]:
spacy.displacy.render(tokenised, style="dep", jupyter=True)

In [12]:
nlp.pipe_names

['tok2vec',
 'tagger',
 'parser',
 'attribute_ruler',
 'lemmatizer',
 'ner',
 'sentencizer']

Now, go back to the start of this section and explore what happens when you change the components and structure of the sentence.

> **Question:** Can you provide a sentence with the same meaning as the original but with a different dependency or POS structure?

In the space below perform the same analysis using a different language. spaCy languages and examples for loading the models are available [here](https://spacy.io/models).

> HINT: You will need to recreate the preprocessor using a spaCy model from the language of your choice.

> HINT: Use Google Translate if (like me) you aren't multilingual.

> HINT: You may need to download the relevant model such as:
`!python -m spacy download fr_core_news_sm`

> **Question:** Compare the dependency graphs between your chosen language and English. what differences are there? What challenge does this illustrate for multi-lingual approaches?

Finally, try applying the dependency rendering to the article you performed POS tagging on earlier.

> **Question:** What is the longest-term dependency you find in the article?
> **Question:** Can you identify how dependencies could aid in splitting sentences?

In [13]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 97.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
### BEGIN CODE
nlp_es = spacy.load("es_core_news_sm")

def tokenise_es(article):
  processed_doc = nlp_es(article)                                     # Conversion
  return processed_doc

text = "Julie comenzó a trabajar en Google en 2015."
tokenised = tokenise_es(text)

for token in tokenised:
  print(f"{token.text:12} | POS: {token.pos_:6} | Detailed POS: {token.tag_}")
### END CODE

Julie        | POS: PROPN  | Detailed POS: PROPN
comenzó      | POS: VERB   | Detailed POS: VERB
a            | POS: ADP    | Detailed POS: ADP
trabajar     | POS: VERB   | Detailed POS: VERB
en           | POS: ADP    | Detailed POS: ADP
Google       | POS: PROPN  | Detailed POS: PROPN
en           | POS: ADP    | Detailed POS: ADP
2015         | POS: NOUN   | Detailed POS: NOUN
.            | POS: PUNCT  | Detailed POS: PUNCT


## Part 3: Named Entity Recognition

Now let's take our sentences and look at how we can extract named entities.

Again, we're very lucky in that spaCy will do a lot of the heavy lifting for us.

To start let's remind ourselves of the common entity labels expanding on those we looked at in the lecture:
+ PERSON (PER): People e.g. Ghandi.
+ ORGANISATION (ORG): Organisations e.g. Google.
+ LOCATIONS (LOC): Natural locations e.g. Mt. Everest.
+ Facility (FAC): Man-made infrastructure e.g. River Thames.
+ GeoPolitical Entity (GPE): Cities, towns, countries e.g. London.
+ DATE: Dates e.g. 01/02/1970.
+ MONEY: Monetary values e.g. £34.83

spaCy allows us to extract all entities from a piece of text using the `.ents` variable. Each entity encapsulated in `.ents` can then provide the raw text `entity.text` and the type of entity `entity.label_`.

Below I have provided a sentence and the tokenised variant.

In [15]:
document = tokenise("Ginny was paid £13.45 to run from Swindon to Mt. Snowdon by following the M4 motorway in February.")

In [16]:
nlp.pipe_names

['tok2vec',
 'tagger',
 'parser',
 'attribute_ruler',
 'lemmatizer',
 'ner',
 'sentencizer']

Using the document above output each entity annd its corresponding label below.

In [17]:
### BEGIN CODE
for entity in document.ents:
  print(entity.text, entity.label_)
### END CODE

Ginny PERSON
13.45 MONEY
Swindon GPE
Mt. Snowdon LOC
M4 FAC
February DATE


An alternative way to view these is through the `displacy.render(...)` method, for example:

In [18]:
spacy.displacy.render(document, style="ent", jupyter=True)

Try using the render method to output the entities in a series of sentences of your choice.

For a starting one, try using the sentence above but remove the '.' after Mt.
> **Question:** What happens and why?

In [19]:
### BEGIN CODE
document = tokenise("Ginny was paid £13.45 to run from Swindon to Mt Snowdon by following the M4 motorway in February.")
spacy.displacy.render(document, style="ent", jupyter=True)
### END CODE

Finally, using a range of articles from the provided dataset display the named entities.

> **Question:** Are there any entities that have been incorrectly labelled or identified? If so, why do you believe they were misidentified?

In [20]:
### BEGIN CODE
sample_ds = ds[:5]
processed_sample_ds = [tokenise(list(nlp(d).sents)[0].text) for d in sample_ds]


for d in processed_sample_ds:
  spacy.displacy.render(d, style="ent", jupyter=True)
  print("##############################################################################################")
### END CODE

##############################################################################################


##############################################################################################


##############################################################################################


##############################################################################################


##############################################################################################


## Part 4: Relationship Extraction

In this final part of the lab we will use spaCy to extract [noun phrases](https://dictionary.cambridge.org/grammar/british-grammar/noun-phrases) from a piece of text from which we can generate relationships.

To start, let's tokenise a sample sentence.

In [21]:
text = tokenise("Lewis ate an apple.")

To see how noun phrases are extracted by spaCy the code below will output the components of each noun phrase. To aid in your interpretation of what is happeneing you may wish to check out the documentation [here](https://spacy.io/api/doc#noun_chunks).

In [22]:
for chunk in (text.noun_chunks):
  print('\t'.join([chunk.text, chunk.root.text, chunk.root.dep_,
            chunk.root.head.text]))

Lewis	Lewis	nsubj	ate
an apple	apple	dobj	ate


In [23]:
list(text.noun_chunks)

[Lewis, an apple]

We can then use these chunks to create relationship triplets, a simple starter function is provided below to do this.

In [24]:
def extract_relationships(doc):
  relations = []
  np_chunks = list(doc.noun_chunks)
  if len(np_chunks) < 2:
    return relations
    # Extract the main verb from the document
  root = [t for t in doc if t.dep_ == "ROOT"]
  if not root:
    return relations
  else:
    verb = root[0]
    # Pair adjacent noun chunks
    for i in range(len(np_chunks) - 1):
        head = np_chunks[i]
        tail = np_chunks[i + 1]

        relations.append({
            "head": head.text,
            "relation": verb.lemma_,
            "tail": tail.text
        })
    return relations
extract_relationships(text)

[{'head': 'Lewis', 'relation': 'eat', 'tail': 'an apple'}]

Using the code above try a few different sentences out and try to answer the following question:
> **Question:** The `extract_relationships(...)` method provided has several big limitations, what are they?

In [25]:
### BEGIN CODE
for d in processed_sample_ds:
  print(d)
  print(extract_relationships(d))
  print("################################################")
### END CODE

The Rolling Stones' US tour is likely to take place in July, following news that Mick Jagger had to postpone 17 dates due to ill health.

[{'head': "The Rolling Stones' US tour", 'relation': 'be', 'tail': 'place'}, {'head': 'place', 'relation': 'be', 'tail': 'July'}, {'head': 'July', 'relation': 'be', 'tail': 'news'}, {'head': 'news', 'relation': 'be', 'tail': 'that'}, {'head': 'that', 'relation': 'be', 'tail': 'Mick Jagger'}, {'head': 'Mick Jagger', 'relation': 'be', 'tail': '17 dates'}, {'head': '17 dates', 'relation': 'be', 'tail': 'ill health'}]
################################################
Hands-on with a transparent 3D TV Jump to media player Chinese electronics firm HiSense shows off a transparent television at the Consumer Electronics Show in Las Vegas.

[{'head': 'Hands', 'relation': 'show', 'tail': 'a transparent 3D TV Jump'}, {'head': 'a transparent 3D TV Jump', 'relation': 'show', 'tail': 'media player Chinese electronics firm HiSense'}, {'head': 'media player Chinese el

As with all exercises in this lab let's now try out relationship extractor on some of the articles from the BBC.

> **Question:** The relationships extracted will be complex, can you identify any dependency between the dependency graphs generated earlier and the relationships below?

> **Question:** Are there any particular relationships which the extractor struggled with?

In [ ]:
### BEGIN CODE

### END CODE

## Optional Exercise (Challenging)

If you're feeling confident you can try improving the relationship extraction method such that it can capture more complex and long-term relationship dependencies.

## Wrap-up
Well done on finishing up this week's lab! I hope you were able to practice what we learnt in the lectures and now have a practical understanding of how to perform POS, NER and relationship extraction.

Next week we will be taking a step into the realm of text generation in which all the same rules need to apply!